# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1fWtz28ay4Hf9irlIpQIkFELKjzjcMLVOJCeuK9ta28k5exQVCiSGJCy8ggcl2uX9W/f7/WXbjxlg8KJkH19XIuIx09PT093T3dMzsCzrRVqm+Vys4ccuHHH8s4j9KPLFg1Nh/14l+8pPHpwen4jvxUm8o/elvC2r3MfLXyKZBDJ3j47+KFIRSJHLVZUXaTE/EuJb8TYN0kJE8H+cBhJ//65k4Cdi5ecbH9/JREBbv138IXy82Mn3ws5kvg4j8eLpP5+/eCW+B0hCPD1/+8qZCL8IRZKKrb8XsoByPkIocymqxKcuiD2CKcJNFcIL6RIa59BQJgErUUiRAYrYeqLafv30hbBX/morEf/M34QJQIXL8zCpbh0RyT30MSUssAd+vtqGO0Y983M/gn7NhQKaAyKqr6L08wBQkZsKqheMyVkkytxf+u/gtYyzFKqLnY+gvg/k7vtiGws7CItVio8AM0d1gElE2Kel3EixShNsauUnAdDxf+mBEPavUO47qgplcqCMTAh1jSrVNEkkNjIBQq7CNGlGo6FbLosqKmmsoPlNhZ2igkhxJhujKqA3MDDFFmkX6dFoyPauKkpZIKUyWYY5FonDIlYFZSHCpCj9BP6TqXtkWdZRiAQqEWHgqRxwddcVMB4UhQFarfX7zaq+itKlvkZMorC+DVN99a5IE32dFvoql/qq2FZlGNV31RJIvpJFXbLc5tIPwmRTPwjjuvL7MAPWlUf6PqnibI/oJtnROk9jcfH8XKh3z2N/Iyf88yorjriAK5pur8PN0dHvZ6/PxAJQdTO/3LpBmCd+LG197y8L/LU9Dxv2PMc5gvHzzl/9+p9Qq0bWPU9X17Zz9Ofzs3+8gRe2Ba0lpTUR1tJfXeNvJNd0n4ebbWk5R0dHX4k3wL15CCw2B7R85JRCbEIQHhhKkYq1HxXYuVRkINfAB2UVL0EIZATMku/CIM2PqH8uCLP3/MXT3868i+f/PDtHFB4/9KbTKf5/hG/fPD/Ffp5MHz6BpgO5FoW/lh41a4fxxpkTLwFj/OlHgBJIepLmoKvC9z4xIyOII4Zv7dKP/SSdoDTC3zUWLVPHRcZCOOEaqgNfFMx2K2pDjYZLf1WD+C/3Q2B+aLeSZ3me5rYFgqwUAfYeiGFgsCP8gIJY9WYittAtAO4W4Xup274RP4kHJwJFhq4OtXVeQ5aomGJQgiCPoNFAmUKrdhwmYZwiuOzWUc1SE98C8J/F8AB8RoObHLSNbBqI/Vsbu+dAI3oEG7DQY9XvVZrtbcd84ZZbYJTEDyPb1jUnNQxHD8P505e//uvVm7pBrInKlUcuASZ+/dsvyLLw89RyhttOdjIvbVWESuQStEiCrxWjrSLpJ1XmpVWZVWVhX0uZLZ5MgXfCxFvnUnqb5WI2bTjwF9CsPqk1Qz0CmcQulO9QU4Y0RdWkU7JS+oKUe82EZb5vkA4KwLkA0ZeBHSDDigA7iUrNxT+1yL9Lw8Rm9eC++uOtd/r8NdDgW8txkEi6VFiAqrADpyE89gSbIAUHiqS49qoCpasNy3GpIMz33377YFrXvtmCfkEkgQmEDdONHRQ48kgs5GOq9JNJMmNAqH/QdgDIpZk9dVpvFEZ5DDO5tIOJCDcg2tKTyI7F4m1eyXYFo5vyNixgyAKY9CwX1K/VaRX/QdlcxulOmsVapT6fNPJ2JbNSnNEPzKFN65kPk8bR0SqCX3GWgF0h+SWynOeB0JaeZxcyWk9wWl8vXqYJzAdRullkeZiURkewkJsBglgOia3QCmQpV6UhW1QQIEBR+Nt+DCRY4gtspv2m2PqZHH/jxbsDL8uDFUdeZz50cPyNdx0myC5KTOqX79IlCskHy1+VaINZcwFSasnEW6WRj3cf2xU8sPZAJm+g0jOYq5q2voJpGw0wEFsZhCihNl6ATZLuwsghcwm0H8ztMJ2twAYulQS34UsQacSHphdo/4mL+CgDGe4fPKIHS7bO4MHjqdvDEYB4yMvGLD8s4gplF00Yg4FbWsREza2ywC+ljeWBK/zATjOQ23arjqEhxpm5ZuhWG2z/AQE+dlnND/Y9mtMrRQovS9MISqzW7luyUS7g/uwWvAfwRmyYWbybNL+WIP0nMI0xZ1xa6pl11aAM5Ij9axi6vOgRjHSDl14rDdIIn0wKsCOJEIaUgVZ+E6LtskqDcJOivQt8VqGFocxmw5HZ+7UTM4GXcRYhL8CbGpxf5uESugP2USXRYSCeStDyB4MbPSGYPUALrEKfrOFCFkDyem7QKpNHywvCFaiL5o1byBL64sPsY1tKEmEqRIFyDpZCmbyjYCODOLlaY8VQGqHAIWkcq6rlEqoTj4yVA5pSC58mX6Otap4HmJ8iaWPwiP0RP6NFmJoCdyPhrcnpFkzNBdG8Mydetotd/TsiccSK7fjf+Mc8XQsKjBKqDe9Gok9QqMkKPLi0WNgwDGA2J+X3RpTAotHpPY53QMSWoJ3DNFrW0YHap0YPnBx1kpVAvvPzqI4SgJPZkg7yl7bVBmbWzdpfSW9bLbX3VIBn7NH4gOTkTRWwYXAGubwa155oeeVoebVh2I5LHe+bFzDkOb3zQrLYRorVsOWOSrkoAijyI2UbfL8DhIsyt9cu+XiktQnUmpvb0fPialSPowcqB+YIsBFs6yVpuqwCCz+SklRRHQ75/dkchrRjfSn7eXrUiAXwSh7YhFi7mQSIPW09uQlhpqN5CIujv7mEJpbVei3B6tkspg5iu+7ThA1QVOXD9FpCUzz12I8fip9+AlfSGSyoXL/lONlVD5PBAgmOB9q/S6ehABozGAhw8Y9hkFFvhyV69ph6Km/beJRp6ePMWFSxLW9hcsvsHExiGmGnb+nZa+uCJUaFoFRUDYRlLj4wMGWvzt3Z+qP47Rd8bfW7trY+GD0Qx6Kcu9P1x8LQ/4ouBNWYTnliycJMGhpigp74RECFZWowBQvt/kEA7iVbiehnKqmtVcZp+PZZlN688EtQDMnmAkBHoWEqYltAo7vKu9gazjUlOLkJ+FUdPlZYrtMINPCC8K0K6WHgAbRY0Xc+gHWoQ+0h+0o8AyN/++cpkDyQaD2swxXFMzmU+v9m01tyD3M/C1UocVT96O654NvD3OCtEfQuiO145fnRJl1Y8arjvShxf7vP2IPvAxxsZbSlvtjcQ6H0FYuoyYJWDzhVWZqE0NCASrkn/E+GrRgWe9nwK1nCzKg3YNiDRUSjrO7I+DEf0ICbD/5Gk7Dck7/WntV+RQE0zURXvMXawn6E1jM8/67G28HgChibGJ+i2CmgBs6cwVoURAW7RxR+RFHWOpYsfgUSYUSYQzKi9ONlCAUouq7tUgBeg1IhaQ6YATZTRkf7OQqLXylwFbQmWR3kTGGKbrQd3rkYOATbpXBXVeCDoirjKnKB4dMbr1w/OAH5RKIdqJQkn1p8CbbFFlyy627pllphT1dh/gtA2OQpEO41xQByU5jN8Ry20FolFrXZBXh7aeJtsgpsNhg8nFAabrl7XoC2Dad8sOWW097rRRtW0zSFZ1ru+ThwkiTiWeAqYS72KK5l9iOt5Tuu63b0zqBW6YQG2H0xZohhs3F0Kh77Z21V9eA4CMvj3cnx9JgIAAqgHzD6DP3CJLhLcQ2wSIsvuzzWHp3PGhseFXA6WVoHxqUV1LnnAAwRtO+d6e7EO+rJONtR5OczegeAFe/t/DykiGpR+sVnMJ+KPd2r++CffDoHDjJhvPuyXNgQ5BAndobFbrEcaKeRgXH6o/aZY9YerUO8efew/c8O2tCQ9fn83xit1jhN0LWqCpxkixQm4Bmtdo6MHgdF6yHh28HxaM12pbwdMqEvsHrPcCaMK4qi2pcGLagxU4tedaYVCfQBt9kBC2a4nnXV86IL9MdpRa44QLf1EFMR9sL+8A21/g0iU18SREboG7Zovvk4wGkH7d7RF70Q9TBBD3sWPaYbVrkt12PYV73btDe4yOKhUxSaH0SJVv2+KAUGO+kcHQavI/2A77BTD27z9ZdyR+4zmR9eojBNt1o+0fbry2gT3y8kCBGFLMG1DN+zk4xGJDw0Q9m9Bi6tdL1Gb8W6mt+BqPbhyND3VlnlqaodZw6l5g67R2X3PH/appewV6DBclyC2oOfgg5Gk+jhWC11hhQZspT7gbaM1nFb+n/Sna7MB85g+C0zh4E0aP3ENEdHXGC3TG0LSGY5w6sYLYOO3tz4eVxl9mA05oVaIIhgKuRQzJ1hFV3biyU637WjqprpL1P8kfjowPkB5tAkIdjo4NVxAtKcFiPCiN3Qa5knMOQ4Ur/+cfpU7IEBwk3TGwq+Rps0D8uYw6+r6vTlS060AhhZDljnOpnHx1wlH+O2mKkVpav7eYsqWNNyDFpGick6VFhNetqxYleQvMyw8PydD90DjrQ7zMDO/p2uVz/gO5SYc5r7N22xjwEe5yUk8kYlFEzAl56dTAT8ceB6OhHqP8fprX3XYF38Y4dxp4groyjMCmnbsx8eT8RDAPPgAVycTKcOxf+ihQ3XEzH7Af88gj8njx51GwKuBeFMNhHC+RGqz35EQDOE+EMD6DE8xAyH2eNBMOhyMtV/o+wwjFcGcheu5MLCcbAc8PcTMDBBuYHq7wRaKejJ1cNkLXOcF0gz2dA+P/erMl35BS6iILiJCEqY4hb8EhVXOXvsDAfGbMpNWYTxBNOrvKaFopRZsYDRSFeYSODlEmyuCueHxezkyYitiCBWYM9cFwutsfjWupqo1DhAarEBqJQd4hGeVomCUWy7xm3DqMU+WW1zUJ3vpe0cjRg+GMIBUxkmTRTl+6iKT4rsCw08xky/VKRxWIaBNj+N+O3QBAVaFy7NpJr1nXZpVwIHJa1tt7pBuKYBLEOawHJaDZO5i+RVd1owX8Cj1/TkaGQWVAt5ZmdQ8+jEq/Ea3FBdQXECP+1YtkZlhd7CwMxWq4Mm93XBfZLbYvZlQUPSxwZHr9vPz10DWmEskXP4OIUOZwW14trlHD1ndWao3rir8csKSq0ceNOeMPDfDtUtV3B3YY7L8NhUmu/tMVXFYUgq42FcceVjKpWjlz8oeD82k7A7MR0TVMxvrWi+/bCL3aqQQWtR5Xt82llrmfJay3//19BaC6+3oPh/2CgIikm+2eV+7G2W31xhdeuTcjO+xAKwTvM014AzpRAoQ5IzqbzlhpdGWvoB8+4OKIa2LY3lO9mXRZk7Xc7Z1NMuLRpiJmgnz6+TJ3o0UPdVVmCm2NoDZyUBi7boFN36wA1RtvVV2iBlGS4WKrOQLMokc/089/dU8RLcTZhVr1wwv4B5foIpdGp2rCYSV03LpoWh/jWhXrubtGhmHxxOa+TBh+kfBArkF3gxXb6TlBKPblmOMs1J45Uf5Gi/Y05LDFajbFQ2EmCwp3WJfTERtwUXS9LkvcxT2xc/i9m05QLgAult0RnNWzA2bmcTsYff/Qxg3BZMwQld+bd4ta+f7dWzAYoREfI0s22EuSe44jsxI7jwa1BtKAOXxBsfJqXdpLF+K2bu7Ilhl/89bPIBBYpRc6/4GyYHMEWYo227gHn8BpQQKAasCndbvnN6y1RQFWYNnIDt2fTkIVhn8LeXDUsTMRH85CGrreLvRiUXWRSWXrGVstTTOF6D+ZJjoIHTvieY6z3BtZWJyvZuuRen4S4MeE/FNn2HeRgqsMd7HsCjkLgbANeOcLUZF6/WaRKkbVdgzS0fVAhG4li+WfptURiT3E7uKLXRy+ZWvUaNwkQydAi9c5xROWsJAeJVSwGRvS63ohQ23/WTvWkCF3IDHAKYABERAz8vKeVERRknxjSDLnA4EZQQIsEURWsTk06jrhpci12tShjygCPcaZGXLMOO/69IxeAOg+pl8Jk9dP0MaBkAg2Nz0I7Tkv8hyEPVUE9gf43a/yCikmd4ObtqkRXpWHBskaJc+AgZDcqBYBWXUxyhf4AgT93pw6uuPsLipKznpCqBWcS2An1ZzHEdFLc0hBuwM2TRS2HB2iRALsmXbU0sx+n1DbGzUaWEgME/wBRInAmpGHiAOql+ytk7IWXvgK8m7cRpkMXtHQQqdYF7w8zm0imW7uLQ1ALvpJ2IiXUQFCgdVrsOAngP8KiBCWHcYTMAcolvr2obGCd/lABD2SodTk9RWQHtHacXQVcCDhC/iFlCa16NxwJo+WGuFBx6ChPcjeBhFlhx32QT+1Qq/04+g3pqXRU7gjdBtYKbexrqz9BhlbkC0RABMUMhar22HRuf90oNotMrPGpb77EcWLOtlzEyUv3KRTfljSw7M2oMQhkEHhay22WpbRd1JHj+BVPaZSL3QOC7MNl4yvQJKmBStMA9Xd0+UAVH0wNrwgOHI8SYuCdhzA5UWUUwE3jbFIQVJ3C6QAfpZDqdHmjHS3N0hZlRvFW6Jc8z6voU6HvEmHNA286YMs6g38JhgHZlHk71yn3Lv3ZMdJS30EV4RZZOTLQ0HqitZouOLH2i01+P4fBalMZsIAA+uCpyoKu6u0BhsCLHRpohMzV4qN11eOvx0k3R4e/RClGkh7tXfpA+J4fD3DmZL7TrkcMi7NyetCaxWqfQZImTgCEB4ufm/XyIAwxFosS40VIw1y8ajdXVmkToxqSjmCnqMqXwMHMaFTjGu3S0C0dXBhOxqaArYACpzLuekzbkb8dVVIY4e1zPwS7AieNa2STUkhuWMoZBZSvEiLQ3k41efeaZkuAhfWaNqQQE5EXHA8F9eHBVp9ipFcfDSwNXjvirayhpXNSKZQcFyqU1ItEmyAONt1our7qx7UTelnbWrGhwW92ViWacMcuQyMXh7juXQAyuLCitYaheeajammp2lAYyEKo6cLfQTKF7UENPHLbg9BPx0wLMXqbJo89b50nLrcw7CLdp2kG+35fBlR8Ft7P603qKpBpe+aFiA6s/dWIkvarqvZwDYV4ZZ+Wek8TtrqPOYgVIqtEm+pFEXardtw0ffWLAncYGLpxOpvEXD7orVdaKu2+GA+8GB42aTf3IPNbip87kYDB+FKbWeF6B0e4F9cPWD512EH8cscHgPvgTQ6p5okb0i1i2KrRaa3sV723Ztkrho/XwuUF5Bantz4XrGqb4jwVQqvTLUpvVFkaV9U6hAWujv2xQ42cKArkyba6/wiQSfnR9xfMNb+cd2YpOEw/NRLix5to5sHnjK/EnZyLlkrkTr+11TicdfIeHRvgRiK3rOoJXOMFmwT0YIkqFH4AhILsOUe0K+RQsIirWklBo1xKvacZjKcdbYJ7PNuH0ERsYoWulVqFJR3gCe084p4g7N2DN3Yl8C8PGYePUtS/D3KlKd0fOvpb7tgGTZqXpraGiUcc2oN6fGQq15hE6v6BNN5xGTe7oB04O5mhs9S7JaxfYhfSlM15oB3PCcl+ijXs5n4sffzAouG3ttwyqOCts7OKEdnV70H21nWCgIUX9rbuVt0EIrk+JDcxOrhozMK8SRT6O4QmUFr1pGARGXZLMqGsKdPF1P2jmHYwIdgP9k77loBph7a1uyJ5V18DEGg+YqhazkwcP21C0kl7QLj5od2FtK1xyCgPcVrelXWeLmfsEXqagmuRiCgXbgw9F04X1qw82jJ9bE36AE4xu2setmTVKQOQSV3wG+qRUmcr0z2UUyh22+MQ0pclH2+TopEV+vAx8gUp6LhQpEg8DSaEfDRCdMFO+HRUFed5ougWgeiOFVnfjt96uOpxiQ1kqg5qklfxzUauCo675V0/cmG9xcWnRdWuPLY2wKqluuCjfmGXZXeKifM0l6dosyEuWWAyvuFAnIaoeS1Wwuefi9b1ZSQ247pe6Uz3juxYa006aSGPaq4G2gelOYDL6P1VYUpYixbotE01QPrwNnOa3uRE7U6Jay5KjD1wxC5HYmWX6ZgpPiWYlEtg2YJZcsxA9MUt9HIzPD2tH0xFkY9xcW1Dx9UaTONpPHDTUTQ09bKe0yImGAaj1oy6E2oYYsXV6Z5k8o2M39HlEgVTTpTF8qKLxaIpwpcRg0XKsF9q/ZtVGf1m3wf9KtdXm+LCBWeu5xjlHZQf/10qOf2o9xz+TEfqgyqO/prqrrxqdp36HwTSqcJmmkV3fOpNaE/ILvnEalcgWtrpzRqGDxtSQl6kz0fqPnvG10+g9eqpujGnxWmsttB1Mq8FkjOuaQXnfKnmArVNQmpfAV5cWPAIVMGZ9vdZHuuBqF1VC2+p6cA9sC3CjOroHyjidMP3BzeekjIoyX5NCsr7+v1/HXwfe179//eLrN56FCdjXw8cfkOnWOfOgc8QKHcXTah2Pl8JN+0VZBWHqbss4shxKdjIq40SFaKs5C7WpraY/XmII5p0AB25L73b0gnere0T+yVgvsMhoN2hRYyg2Nf9Eq2/nFv5OtknBLa8tsrS9D8lHN0s2uM5T1z7T7CiNg0C+Eq8xlkjaJRKz6VR8zTv10KfAUx7wIDjMq0FjwS+kPucOD9HKwzQ3drwhb5xdqtMO0Euyzy7r0w6ulLuG0qjSVZAZzuojBKjGw4ZWnt4ChIth8Grq/jjFM+iaBr7HNo0KPCfrinWNO7FgOK3t4GW4urbBRVmiilzh2kFBCR5p1hkry7Ke7vzkPSnpJa65iqKCsYnJZQswc4H0Np3wx+TLN35rqZmINxtJ92y2jpPJBAhg6k+BqyB9trhV0TDo+KOJsDvZeDPsKaYLcI9mA45CbTT4OHpLqOXjst8t9b9fnNC58cMSWnzktCmYgS8k+yRcJ13NBTBaR72d7YBoHQrQ6TJNEd6ObpeY/FEucKwmAq6LxciQgYT7YEMkA2cxgfTQUq59j6UEpTXXSafwGoz3KBraGIT0KXorVqpdklqDavUZQZf1ISRXGPCdmbOFPh0PlP3qGvPDOolUhtX3ABTjWYLL/P5cH1MJMouuhkiRLXlbrdUJwukWBiaYLnLHJnKdUvpUlU4XRomb7NSpAS3VRyrT0Jc7tI2KIR+XrKAxCdKxQAp9BSp2TQwKdAJRUTpjYmiX+y6eWhzxRMu63vH84NRC+bG62wAtcn12TaDFGtlh0v9H4P77v3g3ndWJ9AOY+6LL897cDE/T0syBRZnP2Vo4YD7qwxQGhq5W9R2NVYwrqHrErPMwzkKiP2aK5TwU/qZ6J/O0GKDvV5jKwCcb1AcfgalEhjZHdmlZzcbgD6aRKrNObRd5/nSYs+ptf2ptn7mtbVU06Snr1KP2Ft1VuftvATuYHGCs3Q1W3uJ6GaGtAoDmwuBPC/F4iv+Yue5aB+Syzkg7rrxFjNqmCpl6Fo0B0cHdREvL+WLnNvDgEoHNxdEhVjhxcFk5p+7BCBdZhaulMNAFbkjx8eDdAOoXwqajBnRgE8/vBdsIeCd17uRRskSmpGYsImSIJ/IiP32oOWE++QjNfuDVWLimxq3BzVX9ddyBNdwvwiX3XAge4VzCcWj02VC1/HwVwjiNjf2oyT9UvWPxtyx/mxnN4jLm4INgZ76YzZujTvZ4jhmYt58+uzRm5ciOTX4ZABbXylDlCajFJLMJm7E0EdU26ydqYOsiTEo9IZmh+OdPP3XLcWu20Es73UWdIcEd3Kc3QoehpZ1mauAyd04MGu4hjmN7Ry36hO/BR/5c5huDdJgP9YC2OVHz4cmcBom2Kd5/GhjdgXDv1d77Zei3MvXvZ/wO2YIti9Fw5dDLe9T27r5Tj5R1cNSXNA6VEBu1m285fAbo5mEDHJ4dGc6wPol8XUk6wi3Sh5RjyIBMaNARvCbbPoec9pG6BqRTOtUKM6LSoj4efg6OIFgVxKN+fXDcBO0MMsl5jw2eCc0xpgZguRw7iyXJuqGKO7jUQPKiObmHl+P4MB08bSdEB59Hbk7JpoyRSEKFJW3L3K/MhFNidFoytbiPuAxK9djzViA4PUUXuRo99NQMtnUKaatMFdKxtyaqXWAIksPaeJYjHbQIHiFcW4hUAJYZXjNy+khmxr+JMsselI9tL1fiZG8r/7jNhFwX+Jl4EP1u48lE3DpHI0YDajww98GJ/wF0Hd2jE96qTe54C5PdOCa7Hia7A5jod0gsKl9sTAE556gQSAEFuSROmKAnQ58iQsUcOUiF4/hYdeEv8zBvGMuAJRN9OK32SyNfCwca4e6XCQDWZw3aLavhgIQA/csKXIP+uQjd+BrH9Oq13W5wI18Ny+b6rhOhoeZAgMckA5To9rPe/GeeyukW1RKsYLvvqKtSmOSdTXjp3Ajq62i+Cn0bwfo6RN9ZgdQx+DrWjgOj1QBJSruCti5sNT/WNrWaIc0VSdb19S26EGbS/bp3xjMeKKGEuxbsbrIFfiFCqaSXr2hpxaejOumjHBu0e0C9y8Q4vExkEe4p3IuOrfIVeJxYBf1I8a/nF8TXQc3XBifX2N5voP6nBmt0wJhmE9YnXQ92yYdzr6XLJ8eb+9K9ZdQ31JbmZN8Eeo052Yz+6jl5GY2nyNTZCfbZRAwcCA1duLGcT994qB3z9jaEsDe3su8KpMcMsrR7mHVfkMNuxFg10jnVGoo5LTx0NgZ5guQjLnBPaFTYDh/QW/uOqO8oWY8000JFuPQ6mO+F/oKNpEl9BJ9HRtaCTmSzSx1KmY1EkepaCmJdj2BCraaEGs26xDK6D1jceqqrtPlnyvD763GDENVXWhactsIf4llcXFp8RVviswofUJpnQ++Gp5Coiq3CjFkJJQ7j+Lg1n1MZPL9YhaFOhVA7phe4V8tYlcUtN32+UfTBSAVCRvjuTR7COKNoOUOr9z8+wThunPmgHNmtA/1isNx7YtGqrL8N0LGUcM8MNf4+G4A/I8PoHE/7aJ2UWq8pB2G+4OphtnifTYg5F0yna7lfXGtdw1+3Wax3+gHZEQs2JzrGCjgNYbH1WPfaULMjI4Puzk4pn6rsa59GXneHBZbb7IqtIXh3iOYhQd85Iz7abgjozhk53zdZt47p1k4Cn9St1UK4Yy3QfntPiKAoqMbllQPYl7ihbTQu4Htk4sAPTKN1qwYEJAW8VRY0ttUrMorWpVUrAeYFmht0K/rdSJC8+S4GErNXpCXUX0KmDeG4JO3LyCbr/pGK9SHR91ABoL/pyOjtSARvqxSE9VfyV4LrCruWpriHsH9CKLUJo/J4DARORx1+7TiQa3XVPXmoY8Kqr6WwNTz0wRRzSl0fWA5f72j7FzbqwfQYYTaS3dIvhi3QC0vdF5sWfXTGBHQxl8XYOUidVUMsSc/+N5IpXMWy3KZBsyfQHDnjLIoYtTuoeFTucGO11+rUB7vcf4XZM/i1oQQx+KR58/zCOz17dv707dkpf+kB1X4Etl60mBHjve+nTZGzpL7lAxyMZ0FhHgrZ3P1xR6CIpNHim7evXp+d0tC5eP4tImvblhtnD9HtIs8Jf5UbxsYtfgHor+Fj5DBeMNSjXun3Slj6crfGw4MaCnCuPN618jhyCQb+ShESZUilEf8elnTAVISHhmDgQ8eJjn8W/g6Xt9Bhd4GieAgV5kqo0A/Vf/H8/AxI8vIMPxz2odmrrpyjubhsgtS0Hmkp8HM6v8Kvlxe50LD10/afXDzXqQFDlhFBIfdclxuDBMjf7uvPGRC4JyY4ELRVFRE8LjEKiKJfud5sj5AetPtHBfi0sVEosZ/xRw2Vm4Jg8ICpBox+MUoaDioEetHb8/HcsYAo9QgROuMC2KPfzn8R+vW94d3IpeJmF8+kshAInlWWp3xINby/PzAwVtz18paAdZB79ss/7wEnC+V7n6j9A7LBBd3yJw4rIPh9IOiB/wEH/gUGqGt2uHc/ohQ/weI+eQwg/uSdCrJgsuCS2vmrU+fewJSOAH+ALNan6vN1jShajrFQotzYAcn6E1+05EoWK3D8hxEZ5OATA04tS8zDY1BUxJTYCiNbJAjYj9c64IuYENLYpabcYXibEPjrweMpDVQPXP12BAhjrCn6JwdCtCgqYn5sMviV4aJP0/ncMIQRdhgwrXTa+uMJx2wWuDPP4kiQTl2vs9Upebx7kP0qxuPqLlVe4i/nZy9Pz/CjOMc01Rwfr/1Vmeb7Y0r7qTJ6SHHqKV1e9NfIhgKOOnCDoprtyZA7PuYGQhxdIg/cAIU4mtWDCi+hsqXoB3dMQotO/bD5hsEyTdULvnGGwNWhG6a9fkIp5ly5vmfAOqGcX6q7QdD1mGEXZzr/hB+pHBQm33GjognbsTRbKKli9g3E7gaupgAZYU07A8CIWSzFNIRI840mBZ6TP/9jUc+79TGO9ZKDbqFZkqAjCKhQM4mTd6Ia6rxqFjUaswKPm+18tSfL+dBYtWPevSBnATgXOxAAtyzMl88vzug5WKTm8zdvT1/98fauFV2M2ai9H8tqTacMzIwvQmGO1aJzpAsijBZglruMTS891XpKx6CIde7HkgKsVEfV5Yw6NvqsZ3ju2oDVGJP1DH4perB23oIo7L+C7xwYOYQ26HvHdTxYKQU+eJozfRe1+h/2G5ta+PFRm5Il3QdgYtMuMxc/VJDZMwfTJR/S6ZUDWnrAJwWNU4ZJx+WhwdenxhBxcnUsSn8FQCk0PgR93MQwyT1uO+hSQx+UVA3Z7WyIZt1m79+xdNOFNHLQcI8ZLlHjOd2x62OoDhayr+V+orNP24cMxcVYsAWcF/HzgjkbGwIYmhbjZ0orOaCDbsZ32RqM002JBVmhRFhD7NNNeYsp0eC4Wzx3EDO0gsUZfkcLNSnuqEPdNB3J62dol8cPptPpvL/vkV8fHR15Zy9/e/7yTA8IfyEW9JUn6ROedvf7nPhpVj8Sqpr+VK2G0tul0oA/a+DV8QiFjAb2/wFlVoP1", "blender_rig.py": "...", "app.py": "eNrVfV1vG8mW2Lt+RaWNu2xqmi2SkmyJNr3QeDy2g/EdxfbcG4Qj9Ba7i2Rbze6e/qAkEwLykgUCJLhAXhZYJMguECQIFhfYYB8SZN/G/2T+QPITcs6pqv5ik5Jm5iaIByP1R/WpqlPnu04dGYbxJsxEMuOf2JWYMvNVwj0/6jJPBEykWQ43I7gJsySiZ2GUiWkUXbKIudEyYjGf+yHfi/Np4LscnoX4f5bwVISccQZXK5HCpywPWZaHAAKuXwRR7s0CngjbMIw9fxlHScbmQTTV1x/TKNTXUaqvEh560bK4E/oqFW4isqJZmk/jJHJFWjzJFomAiYXz4oG/FHtFxzRpxlO42tubJdGS2Uy9g9nM/Ll6KEKYrdCv5iJz5JO9vbPfvnn7no3ZxHjBl4CRxLCY8SJKEkFX73mQe1xfZvLqS+4H8uqdiKM0wqu3PIxSvHjNp/jyYu/83cv3Lz8g7PUeg3+PYA2W00SMmBnzNEotFrlZIoTFXA5ot1gmrrM84V2bDfqMmsBiLQXABRjLCFYxYh/yZBoROIMuDYB2bLHDkyOLPe7DP4sN+0cnXUu2+UpkPAi4R+1OLHY8GFpscLzZ8B2PfdlqCOCGx48tdrTZ6gwwgG2O+qrLk802b/m1v6RWx/1Gh0f908fQ7Hbv5T8/e3v+zcv3zldv3sHcjAMkPaDVg9d5eJPz8PCr3vCApymQxoG45ss4EA4AnYvU2Nvb88SMOTM/EKnpdUfUa5R4IqFVlHhyeAjtvcieB1NcFPUUOKXxJPHn9mx6XT6xp4EIvUoLuSr+pwIadVj8082A53jmu5EdTT9Wvi4eL7Og8hgIMRGpH4WHnp3imwbQ2BefeKrHGgugBX3DE9eHJdW3MAHHD2eRjYwHVIefz5BgJvBJzLOF/THyQ9Oz2KzLZlHCZswPFb78GdONxLWfZqm5+U23APkFwJyVIFLgJOGZyPs2/tj41tg3ut1udWLQ3wwFEX4OAEEm0N0M2NNLr/xsYZqGHURzw9LdgnDIE2yslz3Nl0ue3Jg4ZbX2HCZLGAC2Ng3+MU8zgZy4vpWdB8DnhJBiKOYMqBRwiBINfr45G7H9/TXCmHSIFx1k9c7FyLqVvLm/z376y38Ho/Fy1/dQPBo4mbLX8iujy0SQCmbAxRdFjzNDdUDNZScdi3X+vNOtdvLjf9dCgK25bAr3ne4ti6/xpVEBKNIfchGILGIVwEAN2NqsPFrkSD3QWR9eyBuLEUUxEIY1kJWvYvVRpwfwuth35WUq5jnI89TJIhAvah63LK2Q8cz4OkqWfMTWnWUeZP4KyIt3akirvlBI6wyAvrKoc1sbltnW9QzBU680ug8KbbiWtSl1Ur/RrcKw45fdhthlazeqcdlRFfg7oFuRjGoLQI804moI+ekP/4198AWooHTEaPhs+8RKnNYmw0yS+t3mlPBhMRuYS5WQdsyq6KU5sy8Dmkfrx9OgmGP7x2uzs/I9AQTWYV+wNEvMFiiyRacLXALNUmYKECnyJYsDUKaEbZhTbaIbnxdTNmq8RoR5H5Ld31drJOVNjalJh7ggpEGcKFlTyBObxzGgwTTOykZAeTAZFMpSBpJAqUG5UPKw1k0QeQ4JgO2dfPPtV/BkLpJoo5Oa4gDcXyIhrwqh0jVIYl9abIVSVw6p7PHC9jOxTM1u28CWHIwQJxGBL1Zi++DeyQYjhu1RpIZIwaCdmYu2G5qfqRS0iuyMlr5WIgE1KZA0A761L5Aqn/+Yor0KALFlqphv0qkBkKJ79fm/ymfU+vP/DOHDAxcGebDk4RyMRqOudQvtG/hTAZZllEoFAYLSF0kIv2/YNOGfUHIC9hjIX5grUC2Yt+k8F2HbxKTEvWtar6mVnlUxqcrHnQtYWMRhZR7Y9ZQDs7hCWvAAFKwD3m0bCEl0MBXQzkKLddBlz9lg64jOCw3h8gB+glqWKAE+BV9AwdEDrQGnoYIdf33Dlp//FshAJPhpZVTlcqd+6ODHQGaIU+jA0HS6e2gjMJL77DdKhS9Jnac+rALj6ABp3bhlibVSqdDN5khwHvo1K57qaSBbXemxKoEByoycgMnF9qX+6a//5n/9jz+w9dWtAqRsHOP78PtQMTV90gWb592bV+RAGGd5hta5wXNaPCAYoGF6sqArEIfkuCQ+JxaDF27OvSSPhUcfvIeRFmjB1yE4VTmAuFWm1coXVyJxFtkyML3xb0FeWWwh/PkiGz9G4x3fq8dXfiLGX3OQvBYDByPm6vkyndOVmjx4iC+lJ8oOv2ImunICzNQukhA4lf4s4UsBNJyAMK9ICZBMMUAlrxWMZDAS0PRGd1NRD9qMHpiwdHGHAWsoX9jGeYE9Wi6LQjusyDPPX4GaugnEuCNnPFrL37fx9VPPT0Ef3Yxmgbh+ykEKhz2SmSNXoAP+dJPCDDRB/dlNT3k1uiVC6HmAOzcDdTByoyBfhk/nPB4N+tDRlLuX8yQCNTV6NBgODgcnbbCnZLr30PHN09HgCD4EQFEyenTKTz3On86g094MfNngZpTegC287OV+53kLqOq86avU/yRGR4/j687z//0f/8u/fXYADZ5jq+eoc0xYXsS68T6fYlSAk60GMjHOg5Sztg6mGrwaojgRp0J0nr8SIKh4gmSBYuvNGYB/qYnz2cH0ORrP8L0cAP3UXIfufJ4EIKJhoZO08Ol/yKNMUJMfgGFmRjYGBs9MDBfY+MPssn0UGWAFF4SEJF0SxA/o4cyMP1uN1/iibIbk3mhm/BnxwKBoQ2yw0UgyByKPhmfSvZxImrg0zgMZw3DA/z5An3a8bpJwp0rCYLT8+foHNbYKCWtmStxxZw0/bzsa9Ve+ly2Awvq/ebpJ3JKaRv0GpbVR2SZ5dkDUBtEVEE8eBCkoexE+ZddJL4Xh+zzoZQl8AlKm0wAuPys/ghWWw28QaZ0z0XjoEf+NEhz+U/AD537Yy6J4dIQk+4wzkDCzYv4ZvBfZuAMGKw8vOw0CNeqUeTI9mc5mT0s2GBwizLNp4icor2IeUiQFg2dxgOLpp7/8q2cHvKTOvb1HrPcL/rG5YgrUuUQoKUB866fLSHoLwGtoccAYQJewaOaDpqegXBE0AbciD6WpCboQx4nq2uWeCLlH4EypKHvPC/cALgu10LXZC2hIvO2JFTxcAUnlpRQGZoevVejwEdgEbAkWCqwzKl0QcRRAA3hcTsBmqHXgdZgufYx2ICZX/ipioQ/NkXEprgLw7b1/+u2XpOq0PnKuuJ+ZaWaxEEgDXNYELBCRplXlEgvsmtACygI6mvKPEQsCQXYAjpXGwf4CQfwFW0Y4VvQwoDVfod1U6JWrBTAfdaUjFGk2MfAerOQiVIHPPMCvcVHyuh6YCbM3+/bpqUXNZmDtW6jV3DGY5fCks0SDAj3pqlDqYeNO1geT1e7P0OErQybULg2EiAHucWE9IXQwqapjSLgPonCe2C+TJErQTkYeAzsSgUNb6JlIVMZQ4shEMi4xmS6iK0fPw4kwuANrm/oYSeZoA9HiAanJKDMuPPeWoAZLtSylsB/CmriZjMldYmivCVqZd9TMBoMtRHtFmDD0L/Msi0LbDXz3sovCHXAPNJVWRd16E96ITXAygFoMZ11Kh7CkIlwtadSo6T5i36n5obVGOohIkshlJT6NkDhcEeQBT5R5D/TloXUZsqVYAs3y6pCqdhMaQMZLIjnFDR4wwYcIKS7NPv8t0v+Pf3eIRKAtIxWo/PEfR4DVlE1BpSL7brFblew6kzFAYMfzKPj8x3mEMWLFimFhD5crrowjCuhJLMSwNpvBvuq84pbwYNwlTtBPQbyirMTHz9kxmOOEe7QAVcdgfcQOjcUExR0CL6MWsVggZnBNYhyYZSGE/uWQ3oFXgOeVcKZzi2CUsfImWmYcjHMKnUNLIWAWyOCOvJznvodMTl1paxZuooxnAi1aT0QbEOkpOAEANaXIM4IHhGR8GggdpEeQ0u0FEiyuwQLml0LFJjYAexSLF+MPSV4RZ2Og/HMtQboV4UZUNACq4OEnXhVvN6VsLjwfoCsw99Eho5USVXOZMM/8lBaG1o9QrZ9sFSLSzgPBLrcgAmXvgRSYJWjQMjMiTbGIPpI/TpG8tGuXbl6xFGUfeAekJ/eDbPyFVhpuIezvHw5AGg7k1yKcQ7Nyo8bstk6nBCwfj/FDGwx2P3NoluYmdXWls0YfqLF+9HFQajPKzqJLEToLcW0+VrZaRj7YDGRN3+5jFB8dqHPpEobSv4L1g4fSDyLpXNyQvhgx6S9ViEIpF5BfF9jKr3yS9eG6oiVu6TPUkBMY6gWONSuc0CjwUKYGgH4Tm3Qno96wX1EO+NCOo9iElhZ1AYJBUuSM6BCYE0i34hsp/YXbTvzaVHcgIIKIZ9C4EtXHd0vZsoQZhQ6QwlyYUnN7Dcg4VfzCqz1VqlY7ynhbGeZVlFyaFThZcjOqMRiCSEhXSxpI8vBny5xdsld1VpVJWgiRvyGuu1oSFeLnTnjo5hNVkpyCH91t0qoinu4e5Q7xdefHSqSN5ZqrO4rGloJPyntkiTvBUYR6jBgi3HXZszF4YxUxiBd3g5GidTyNIlC2pawFdGmaG+uLu4FpgUzQ1E19w6pGrgWFqUdFQ3HtijhjL+kXGkoczO46derNa/CIBJLjRidkz5FDuM5uYvAUu7bjIA84zi2YceLW2PiEgNFHBVhbOgqg712zPpE4kbgvPitfz8BMCYIWdpKGLsBHjSVZsdiOtz/QlamcLGRPYHQOLBSSguvasAhJpkZR2vKGtEwq5rxkcYXfErHatwV5Z9VjUyoUZSx82oSrGDcSOG3D0fao3opzA35jqLAVad/dpKG8dqWBD8hce6vVLAJHhygKctf//A+h0a1DA/WZxx5wrJmCT+VmwhurOTca4jZFSxcoxvn+PjN3ewngqp3DipIzU0bWf/qX/7mxlUyrDtKo262aZKq5RO79LJHhqDQ7wBhR6QgYvVHACpODtCXpHJRo0EXFcbnb6PigrBywvMF1pYwUL3rKVM9ce8nayqhQlt5duD9ttZIVgDGsOg0M5QJ90NsXd6941t60XPM6yAeseiOKcR6FviCPtgh537DKlhNSRG3pidsfsvCHoxK2VQWNHSXuAnyn9P/52js4L8fZWPukKrclNVgUsEcbVJGEhVf4TLVYkTEYwsqY5q4cCpUAAU1w9QCU6TW0h7ZYpSyUGQ6GVOCG9KSMuZ9Eh4/7RpEWQY9rDltX22wI7hM6bh4GR+1PfiyRTqkw8NicfIovWry2T7EShJMLDKsWqTPbeQG6bKNqFYhsUnXVx7bqd3sN5W3VxLVEhUKBvYyPALKcTZNtfvr3/4rt77+VQgfRjRRyRyBlf592VHDGtayROmxQWH4WqYkXUnLKM3dhyqFI51GZXzt5Ru1N0Gd3uFUrnvhgJqBHpSM+W1weWGYYQxDhS/QUJheF5e9bkhJFmC+BMWCtqOtuS2zKZwdgAYeqQRGX+jqiPTzA0eD2YF02uK1EoZCU0DKh7cTUD4GYwag1Z+iaJ4qyZjbaKtut86RilSM8Quq4zewlPI/pZ52ZEA/aOQCGReq/qLcAHFW22YBiilD6lLIagUzwrntLyTybL5NJB6B2Lmqzv4dxV+/3P/ybrf1KQ67Gd3J1a3t/0VxT4fLGIT2bmqVQfsUDAaSDDrfUwSn4GTzxaG/WJB8cVl2soEnUcNi7ZVQAN7CIngpa8u7O6ZIZlfa3333AvD2V4IV+EwYIpWVVIT3FDBqCn0oJWUcdCnk/zEvK8ZfzewlflKMwVUdO1Y7DuUECBZ+R20ePuq2G3r3S3mpSV2+mL+cNsx7xqNfehNcW21h6D/zxwfGFnYg4AMvbNFBLGazIjVOEQKAmo8f9C7X4EQCVy2+KVTZCAfKeJPFXPOMKjYBSWkbv11zCvS240itIGoqHN+bdSCzW605frEjNu9+6qvRAVNSIgwlgyPZDT1xf7LXQ3oP3hnU68iLP/KCMUtCt7UbxTR2WvoGh0JI7pGIdp7vRBwZD5AilDbKWkZ2avFSvMJ/TBucbLAyghM3B1xM/NTU1xVVFFfA0/f/TvvnTmij3M0tq0B5qaFlNC2SL1bFAwwYaKUr8hLz9UH4mtLXztFY5IGaOCzGDrmyeuKLQMXValIwQpzVG0NGMCMz/4uFqCaOVDe2Vn2Q5DxzaLrmphCJQuNNntpt73Ib3Dho8iBWTgkuVl37q8BVm2k8DtO8IsRgp7leUc5vBVN8ZXysUocGT+FNhCjvu3qK+fXf2FhTyamnnqfDAPMIIdH9kD8B2PGjZfwFQ0FYmFhaN0dB89SVm3C3tWCSY6HH7G8pcfHX+HctT7vFROyxzPhlcgNE6n/SBWBq9r+lls5dammUJqtidpdwsl7sLAfNCW07YdFfYM622zPVGSgxmbIprJErn9Zv3Hyj2nek49TzO9eUq4Ut9Xbl0yxa4x0vXxXacjElqWtskLySQFRIJyoH4Cn707f4mXaKALA+LkFlphMC3Pu+lS5/1euCwJjc9GOwYQfufOM7XhntL0iQtur6mRbV2b7nhVNDGzhNBYOIIZJHtJfwKepNht7GbrqwwWgiO8eQwyoHHU2NnsCldiCBQO0IujxG6E+VZnGfqIQYS9CX4OfBufIixNQ+uCsCYAT+RgdprKcCvKckf2iV+bHblhogJVqbRBXLTt5bRLYOY6OJAH7Elsc6hnUUpdvtwTcSIWwF8MsTHIF745PACfx6VIJAenOw6k2FM9dWgPzwisj5Y48fFE0nTxh1KqwLSwP1X4Cr5yZ3SZsWDlKwDNzORDHB+AGy8wh1C+FXwKyAeXitQcOmo56gacNHHWVwm/F06lsQtgte5s+VwiV8ml06xi3FdiiRMZyveT0a9034tGIUArWK+cLXEldCOQE1dlL5AwmeYDKuzKszTPsWI3kUfI9oP9CLGgzke9kplPl3g4649sM+C34DxHZRRuxp7tQ8G0Fnw7949fCN9eKOWxlNLVus8fytnhiEfTLSDO5D25CepxJpdrvHHaJrK5zxDmRXCIhv4EE9/GNzNMDaFm3Zoh4UOzhfv1LmQb968rRLI6RNFIadDSSKnfUkap8eKEE4O1XYgOGDyRIlpIEEqqQi/Hh1Pj6ezPl7+Bo090/gdaBlDy0pscTp9Mp3NdIu9pvWrmuvWg9mp93hagfdC9ueq/kSf9w/7W6F9QPmr5TC2P5pND71TvPzx719o9v+9xV7DbCjL81QKW7QwgALcBbin9bgDup9Z3XWUu3zIGoBhi+VkPBKOSlrACEJB/aUYX9GprQUZhYtJDyTG8zEuTK0VJsrBVI+PTo48CvnTZ2QPQIdVBw1f1D00Gm59T7HSHpXkopEHTdPPcNaGzrEHafZ7kIMYQzk5JWFmrV+D7n4NDzHp55okZZdkm9ThO3WJCt1c10M3MP3T/uiiYWLTQpTRhWdxFNxgdjCLYWBZOu6sYay3HYw5BeNOCHYtJv4l0SWw2hqz4HaMZGbIlj1KEYT2Q/u4jt6B/fi2c/C84kcTRVTGk8Y81Kwdcw+3hkaH8TU7LZILiyTCYT2JcGtm9rqj1vrwsFMfTufRkA+9w8edW5XzCjPUKYwD6DSNAvBttgN2t4HrPF8jbdyyZ9Pna6JDUlDr/BaTUZ8d4Cw1FlCKoHsGYkYmEmjJUlim0Ak1es76JVnVCJHM0oDfksEWBfpExJ8UuZuIpUFuQaxuutFQpfDetmUUQx8qhJ8SJgscdZQslqfQEKdozBJYbEe42MQ0KCuHaB3tiWd0dT0Yd/oddj0Eav09UPcN3K+RDfv2AG+HtVvNCGoux8f6Uc/j6YInCb8Zd47YEZI49YimGwbSWzKXJTae//TXf8PeU4q9FzWP4tASq8RlSmEG1Elx+X3psdCGZG1lVQ9S1EMPf/UHmTEmj9FolOxtz1mvZuYO+8PhcLrJFPUFlaNqLn0z+7aFxDQVngBgeoAJs5Qry4pc86eNZO9dSb215Ppm2jxM3BW9qciuMLOYsuevEh6P8AelzVOm+hoX7XajkxYMa5sDdTI4fsq6oZRM+aTpCoIIOGgC3u4EYmZnfCVvf6+WrWrEVAe3mtNu45fRNdAz6zMgZrZ+DSSLx4JFshJnlCD5Dt2WUqxvyed+gqcHmn3I7GhEEesDkjQv3a47HUmtpFmQGWEw918jhfbGakAHGizJsO5tI3V/b+/Fe8zuRWtzGnk3FrNl3jstNsdzS2xdyzB/MjgZeOyfSA+Rg1u91/IFuCQ9hY/jPmCh0v4pq5Hh7d6jhZewxYCtyxzvIVIwU5jqqyYxW9cPVOgGQ0Qlw2ap74nGcKcDb3j0lG3wneIu/aZgJsQh09yExz8QLKat/Cng4nDtaRBhGpTqRd7WO5Oyo4ZDBV0hRPZZW5RHc9yHq0BBIuNJj9YKvajTvifmljKMrUdP3CNvNuu2LdRsNqs+3tPGUpGR/6SxwJvyqfH6ugdS3ouuRshiwyN4r8ZxfAwjT/x5Ci6RjQTM1oqwb5l+HvAp/FzXOxnWiex2bxZFGVKiZhJk1VoDpHi1p5j7gefkvnbgtjg152qrLFYJN2JJUa7EpsvU/pKnwowTH8OJziIXY8MPPX+OSYihyLMEHGF6mgZgVe6OPiBux5MS9Ksomgfia3hqyuoi6HMYhWBHTZfyMO2hcY/p7XYKur3oAPnaKSnBIZPUUJwM37a9dzyeXFYalcCQPFugSRpFaG0NNDjdqgmO1tIhesNWRM4lrMrbApBqUp8kBmUKKFK+6PmVrzQI/b4xFiKvOiQpb4rxZH4WiPa+KnsFcZ7tQnprg+1YpxR8JxUgYz2kr5YFcIezQxpks+3ukd4FusD4PeBvRa27EO4l8L3Cbsvo9ZLvbrl1+RuftRPC1kaNcUt+nwpKmqmUakEL7XxCIQNnPjXIJz48UaajrPoij6z42QIFw5dILqlJ9DI2zs4Z5UngwQbc3ocBEXuP6SeMLk3HoIxbBcPHdGyY3fHztRe54JeGmY1EbbsBT9Nv/BT8F88zOzgLLG7QxXiPJ5aVk7p6SO+iK1MEYun43tgArWo09n2hyVuAArIZvetHrDno78ONuP15bVu+1tEPGP1TZ1UpzbrR28MPZuF5Lzzl/OmH3BeJF9XA6b5f0AFOM3XBnB4fWhgKcKQrfTjsW6yYP+rfJgKqcD7waS2KuKWNaahjk8aWttU0ePjkDRbMMYkGxzLRo8gEQJq4iYFYYj8wrHtvzjKqwUOb0mPj3asvz4ziiPDwsN/dCkdP4sx1Qcb6UWgab/FguDwxwEwBtnLgcSxLkqicGzxU0AXwuPfZuqbb+kCCuKMpcR0wfAuaXsqR1PGzY9qD4Y5pF/khYta2JOc0WaQx+9ftj3Lct3cISu5X7FCeJdns7HXtVEijvwcBd3RxJ+jiqySKSWjciYSJPOBhUdo/nQAgtKANI1/IkwD0Btvchw3Ul/hNCafoAB/RTRO+cXGPxHYeoNG2bcwKqd8CJkJV8gA5xHgQzxUnrJn5zp8X3NXI29lCURL/79ADM+moCSZv4e65HPiZrEtAp/PkYDfOu9z5rxCY0gI3dtOGStiX43qhVK4pt8oKbtYT9grl+qARVTIyxqCgVUUQrLeA2a+xj5Iq+kg38ND7/MdpHvDdZ/d20pf5+LEq0UR7OuITdjc87D5snc8CyilOBA8esMaSJyU634OLKRKzbw8tdmyxgf1EnsMb9+3+oEDvUmRJlD5sbG/xXDPOSpeVeYCIrxw82bHoqhwNTZ+y4baUn7mjlwYiYNZDG1DRt08KVJSYQGcpBDXPvYdh4wUMET96ABIoQJRtcKMqNlgwJFqVxfDOARdUFu3B1K8pX3HPSNYfpLR/efDweDAE4h8cg4fd78uSQ8xMwGYH6xFzEjcy1u/LCtQRncVdAsiEyhMy83Gln6eVNcXD/h5XJxV/Xoc0J/peFk6ymaxkOGLHqhqjpUcTUAmd3WQEA2uorImBO9+oJbBcIv7GeoiYfy6XDNMu1DJOcPkuJocX3YcumFpxfa7AjK+7dwz0gYZTcUpuFw/+s9yHBcH4gncXnhSuMPNhF0Q9IVku5GFc9jtZjMyMYjy4wANLliWiVXyQjVk5cdYYqyqGowary5uopm/Pj36W2ikZ8Le4Ye+C9Y6nj0aV8zdYcSmRmyvshoq2CpZFHtIoT1mtgNnenefqGvRKlVELBV9WR1WTVIXPPv9DSJkFBOEuntDnEHetNK0Ww+w7homD2yFOM6UHZOGAwj1qrypjVByyuQjRVvE/wbyC+Y4uiId3Z+BVS5lS/p3MT+1ORicXW+FiwZjr3bSG5wVk5dPUxMPtFMhJxxOyEC9KE+ejgDZbtXDNfXzrp9LLjtJdLuTc2cDsmYsJNv4nWn+Jt3S5A29zHkgAr9BES260U/AhcrG6SCTrp3EsrhJEqqKYztx8AKfICkrpeFh4FMfoeFN5GZlBLK4UY28FusB8KznYr8H1pPMPjhvlYQYGDhaohBUoif5fvDmv5NvfB+lnKzy/j2JwtPcriGA6PVuzTQ4p+cBiddWB2Vtkpgx+pgY5p/rD5k//+j8N+ngElMpO3sHf8jx0bXQDzKZ5fLQxvoEe3+Gw9G7o8x190DnrOnhdWFj+rvcx1H1g+eGHoEGN5wVZM6Ar0Ay4j517n/WT9Q+g8W/z5RRnMDw8KjDwXix9fTxWuFQyZ3yHi1zUWNipkc4CwTOsXLKLadVx9TqCsXozOAFY9kCZvccF1K+hCRmwsngKnjbZAV8eZK9B7w1OAC790PAx60qrUR+leeqzpY/SIktQa5s//n3311mK4iR90047wvE8QUky6NP4jo76hZWGgVn94X0c+x309a48RXwv9anO89cweKSKdcOw1GtDU31JVsrNYLI47cOMp28irPRRPa72IJNp6ujTiXcIWHqMMSLQcQbFo4xSy31dOSa3vad7Ryma4YE7If8i8+hO6Bvq9pzSmjkW3czEnZ/Lo4HViPpdXzxU4+1i6s3R/04kLE8p2oxpFQdM5kjew2xIypFtmcvPiOSL1BWwMv49YvhPqkF73CZvU9pY0D1PG6PcUhNxf1/ZpPv7NgO8YCnZxF+KhA7t39NPleUDYEFyMa8e9sfz14Gs3lDxCHTh+/vaUbhpoYynJy2qpr5HgRs+YMCn99irGBQlDTC+XBZB2C4ulg7WaZCIff3h7TfmRmWtsx/yz39kHIxH4QrQBpVKV/ewxIbVM/xyTEVEaNeoCk/+noNqrY5wrxG2FQZTIy3ONW0fKY5sy0BbOgdO2xDNiuW/ApbhCRZINoH75SwwIe2L4rw/S0FwSlVwLyprFTHbia3Fd9cF6cjX5aDf2I9/V3ezf/zHeymmexoIK/oLEhI3NAZzU66Xm18nO4y0lQNubAugV9rD3Q2mLYaqcuaxHt/5dyBgZdL4XYHepY6bEnnUjxRUBN8y2qax4M0u11CfUmiX9R+jqTJc0Ao0DUMVd9IFnngcBzeOjG7KRO1GQRqYIFgJFkMFoe18bFf3svURJt0ag2pZpSvZge0ueDgXZrVTi+nfk/byTsDJzao5qK8oPx6ZzSLRaCk5ZknJYSlKsiQdWCXfWUqT0GHAchImYFjWQDTL6nmtXDb5E5XUK12tX7WuXt3m/rWL61W3hNqt8glQYLk4cqn0CuC0LvD4QujmCVjJ7k0ZJ9prDTyUDelIDboFWANJJGR/7+9TkU0JviF87TR38dCaWa3EYzEa3UTRTHV0DxkXe0DPRMCq35KYf0lnZW9KUig6rooaS5W4g2eVU6GVxgFfTvHQpFkpfmCVJ2O7GsBkjqOn6E2FK+GhLQ8Dm+X5ef1JOcvyg2mlayr6AZCnmkHJpbC0/X9Br+BzSxrdWxamcgi1Ars4ZqsHQ7Zu9XTYMpKnvX8+AggGuDS5MEGictCzzl2EuuTXDglrvROv5CeC2vvlJY2pdmWy57x/+e53L9/Vy/vOQPY4mCZp4o/yUNsLH/9cAJaYpbK+uIPnisDjBAzNooxKD03zlGKJYGmLJMOiMNNkoxxtGrmXoqyUeCkrR/ghJqniBsyVmFYtKoyh6LFSwcTLliIQHtYxwtPs9VpGrWVPCChQQJSKhiOz/cjjxnn9rDkqI8vDUODebjk4PDrdPrDMVqVoMz0GREWMqEhICSKmLEb4+oId9buNbC6JQ1v+ki5Aeo9pp/bUx6IYRt+m/zCm1WKG6mKzbcj59j1VzNmEXasaoiHA+BVpBTwP3YUjl5mmN35y8lhPjCY63qC/goPgXZmpW/KVBGpScn5ClfnGlampx9SXxKfsXR3dxcMmYltGBAaqRZg5sraeg8l8xWfRlSMQB+qBzpsZtMUzKewtPIcS+8eTZmmA6kaFEhiaqiaaJagqJ7L+piggnnWLP//nFVnM16L2F9wqTYwdtTjgs2pV0PpRbhBRc0GpOr0f6Oe3uHtzvSObxFhkWZyODg7mQLL51HajZWUk1UEdgDkheCrSA8yPTrMD9N9R8B5sd8eNyvc9QFR+3eNL7/ERSlBK9mwUToG5uguQ3ThLi/WjJ8fHNekKj+ukCnII/yZgCpIZtAhMNgeCGhscD21q+h3Ui0zp5vUCH/hko5JtngQpnwnztPt/kwGqNM94ni3GJs7KKsbZsreNzZwlkAEW8mzLZL0/r/wi9pCKkbxW6coMj5dTo8k0qD1qHAOIhluqf70hFyWf1DmoWqbLrVc0OKeiLxMioFLeG71eGPXwb8DIwifyEXR7Hz98RkwCPDIYPqF1G4zWuFS39wljy4oD4+oI35y/RBsVnIuk+vz9h68AqZXKBZV69ljYoSzZVuoZqr1fL+aWsWfscb9RaEseBsSeVAkEG8kgqNf+IAMU6/sJMAUxYmEmhXiY8N6nfu+0d/HF9zYsT7ka36PIwKgndoF/5MTYqIO03FREcsGXNqaOx2aL9w42CS+LvW6rnqpNvglV6Eb17KB6rszzYkdx1TpRKlq5UHi6zzl9WRvWKP+2a+M0PgrfwsrAKW8UclQQvkv1H1cQIda6YvKvy0aVvxxAf3s2kQm/kW3bRsMEbtpK7QJpQyiRrJGCoF3UbJccraT/cF1akgMNjEaEkrdliTbkhlIL0FpK/nLge/8H9HWWfw==", "estudio.html": "eNq1W91y20h2vtdTwPTMELBB8EeiLJGCvLassb0lr72WZydVKpXdBJokLADNBUCKMs2qfYLcpCrJXWovkqpcpCpVudi7XKzfZJ8gj5DvdDd++CONZyaZKg+B7tOnT5//cxo6uucLL7uZcGOcReHxzhH9GCGLR26Np7XjozFn/vFRxDNmeGOWpDxza9Ns2DioAVgOxyzibm0W8OuJSLKa4Yk44zHArgM/G7s+nwUeb8gXO4iDLGBhI/VYyN024ciCLOTHp2k29QNh7D47aqqRnaM0u6HfXiJEtmg0BqPe/Xanvds+6DcaExbzEO+Dtt/Zw3sYxLx3v8M6/u4+XrN51rvPD/gh53iLpng7ZIc+Y3hjnte73x10B8MW3sQVsAwP/f3BcufBYiDmjTT4FMSj3kAkPk8aGFkST+yB8G8WEUtGQdxr9cc8GI2zXrvV+rY/YN7VKBHT2O/NWGISpVbfE6FI9DuosfpDsKXX3pvMjfQmzXjUmAZ2g00mIW+oAfucjwQ3fnhppyxOGylPgmFfzHgyDMV1bxz4Po+XO/czMVlMRAo+irjHBqkIpxnvYxTETOb9kA8z9ZRoCvHoB+kkZDe9Ycjn/RGb9A4wyMJgFDcC7Jz2PAiMJ/1PjSD2+bzX7RNk4zoBKP1vueOkfLRYQbNxaikSq6/41mvTQUUY+IaaJQHlk42E+cE0VbRNmO8Tv3cnirTOZK62MwbTLBOx7QyyeFHZLUvAnglLQPIKlyHlYvfW2k6PKhvh2WhjFyWRIB6D0cA0TVKgmohAcuJ6DMY0sI3He7EoeaCJcsQKSfc7Xme4yzaEvrKmB/axQcj9hQDaILvpObvdfF+fD9k0zGjBZEH87bXx7PulqBMesiyYcTl8vM6UrxfBFhp933AiHk8LAcci5v1NJVMa1VLK1t6m+b9SB/bxHAWxcha99iHNRWze0Na235qNS4tg00xI4h0x4fHaCQah8K4wS4MGs9WvksMqTF/vRYac8XnWkGbRIzPatGEJ4HNPJExyRvFJE39AekUUrx6QzlThk1xSaOkdKlgQ3xvTkVeOoIa2aGC+KI1YGC42rAN8Y1lPihGehM8nt+iRuCrEVMVBw0Rx4zqXRwtoBgIkRVt8kproSVuTbqlLPCbrHYokUnYMneZ/ZzYwY1W8z4qjka7IDxLuSfQgaBrF27xX7thIY5RUD/dn16CQxUGULjacIElm1c/1P07TLBjeNHQM05hzFLkC/Wq7q7iqXE06a3bwky5qjShnG10IdVsFKcfVwP3hcAhUQTwUW0SoogmClrT5faJxU6lIIxA0uZJ0LsW9Pp07N92208UuIR/x2F/c5lnkeq01j9Zsf6+7bvtVq9KJgOf9YtdTWO/acVZcYnG24iyGH8y2a9YWBdXZA9AarWWOIFjkHgiLirRiw43sVmgJYsla7ePuw6i38TSIkazBx6xQt4Wq7Tq/Kef88LvLHY/FM5auO9vfRNwPmGGW9re/h4NYix1DZi05Z0iVDnKfcFBkKgcU+I2NyF+1icNV+ezKFUp3CanSCqU/2zeounh9zgq+donPGCQr0ZDGtaPTG+y2FHQmRJguVn1OoaiNuQ5TxqoHyQ/V3aJ1+lRpxkZ8wPwR3zhbu9ta41puWSsA6ydb7hw1dU59lHpJMMkMyvzdWhBR5h6xSe14oV/SWm9Ry8YJ57VebZxlk7TXbHp+7HxMfR4Gs8SJedaMJ1FTAv2m5bT3W06rOZgGoa/GnEj405BjRc1WqJo4tYjT5s9DyecsQpacNj+mUbO2XOIQknicoqmqE0rN8QZTNALfrYFdKCwMQw54IUtTtwatqslJqlNSOQ0AJQ3DZxlrzLCQz2v5AhHXjt9BWaYJO2oquON1eCShYNTxm4B/YmkBtRW1ipgEzFOR3ooRm9/Ujp8kXhCGt+87DgATZqy65VETp904NRixdcL3FTdCoqpWbKBnYXa147/985+Ms+kn42//+N8lIRUUlGds42NIhaMs5mp5Vbf9IAQ4hwUGIgGkfjJMeC/rDl5iFX7gGT2OZU/yx1u38BMWIVnzWO34GR6//Jmeb4WGvceAfEM/hpmgdE1ZYmRKD0qyFE/v5KzHomg7Y//nX/7+X42TL39GNGA/m7ueW2vZrdrx91T+8O3nAEz7QEKdooAJfXYr2KGEOmO+MIJPf3TuYDuAGxVoBCbnVqy7Xbt9UDvebe7dCtKyD1qk50kwuJ26IYqv2vEJq1rgTzMeWe12vp/OybFBnHfyPMehBLCyX/6T+xmKu5K+EYvBE7g7Hoq//enf1uFKT14zpP8FsXcXWuTwu2XgWsmruit5j+5jlLncZg65Jc9cjTZ92fUYQ67XvZbRMtqUC2nMBweFBynPLWPN5jjFzc1RFZJKh0xjMhbe4poKTy0D65oN0HgishUv/QSRsYFBtukEpCFm8BMnsJakArCB9Br1BexehF/+YyTiLR5aCvKKh2RUf5zykGdiO8wY5MHG/+EvxgmbrISP0kvnPytRWIVLHFhFYOOBwVLj3Yu3p6fGMBGRUZdhsd7P5xevk0GQnUCUCRi1rALlgdbTk80VUATcCpbnZ+++P4Mmo6LYhiKUU2mzBFtd/uztk5PXX7G+AreK4K0Q0Wk8CxIRw+ayrUh4OZ821xYobDs4apoZ37ipe+wLbypnIKfk5hyy8jKRmKnV11Dfvzw7PXcXcOu9i7oy2/fXfOCMwkHdzgdIR2ESq4MqEgSf8olLW+YAQDORSYAe1NGehinc61EK7hhiKrrrwXFANKBaZ+8ZQroaXeaU/u7lyam7WKexV39+9tRA7OaJMDBomAhtnjAeGr99c/rc2n4GtUiPGCaW2cZz4YvMNt7JdO1jWlkJx+MMB3Os+v7p31FL1+C52hvmD3GQ3djGD3HCWWgbrwJkaAKLdwy93BmE8BBY/JR+OaK6GrFu56QiLw0qG5XASCgohgtHDD4C8vXT3wIqk7FcVLZ9D51KeApXuus7aRYC9PzdmaFHv/xXbOw+Q3kSVcgIhd9+321pEs5eP8vZ2m19uwLVed/uVqCi6U0O2e5+K2moaECvrnJCAypujKfQAeAqVaFXf8UmDEHUkGOYq+oEiIY3N6hHwoiEVd2gtRgwaMDAsRAwPHk0k3iXcB+vCR1wFvhcvEcaEsQscaLJHpb+gcaUDshFBdgI1rSLbLsKhve//qeEAPffTygZYs4kHgHgZYSIFtMB5KShJuvLwg4TKXaeuDG/Vj7M+ZEPnp+91ePmgsV0ERCwtJclU26ThHgy41Dka0Svp9PhEFU8TS1htjk6FIfZm2DOw7fUfzNfsWzs4HymumMoZ+yOZfWri84R68wgjnnyo76HwOMLaaVV9GKaTabwkwim59T6dRXp52+fPy0HS8Rw6hySnIBgDfnk5PT8+yCMAu9dOVehRAZZDDs8lq1gl07Y3yk8FlUyDlZhwckYlZRZLPXh9EJOQIUbS1HB8gqHz+ndtPpy3CmzhAqIPIXZmqt7lALTJEp4VAF78+rt6avnwJIw8pw5EdQzInC4Y5RGrlzlkLtWO9P6NedsWjZKuD3LUebO8w2hltXtECAm1N6b8RMWYVMTKWQprWZFWoSubXdarYJ2LwslqpUQZ2IDeyvvAK6Z/wxlJUlOioCGqcR+izwi49vGzicc8mo7nXzjURJUWfscry94OIFyt1v2HnLleYfvttVDp7PX8XLJIKiZtLg4wjAUomoqr3g6NivsQTXCn3MR8Sy5MTstHN+yK2LXSkUFFAvN6u3C0pI9X5E4lB9RjunM3YY0mzcvm518DgUUWK/wqKOXhEqQgtIxj4IKoS/wmk7GYPIZCQeKNZT/4ci7+G9vz3b2q6em5aX6TuMKqmd5g5eFa7g6wDCNHY+l2QqJGFP25ERsQgZOhg727B3Y9L/qvoAttk2C6O5tDw4Yw7btKgIsAoIdRCXDm0JU0zC0I7gb/cgkitRdLBHkhXdVtTh6hxkgx66MPhXzXQy+gD4d2JRWKs1R2CiPJlRsxgKl2+c4WHEAlVHJ8TIpM3Pb9CkTkJOVjAuzcpw49Ix7CGrJGyiBWf+lbZgwGKRNibI5CrNhs271FVlyh8rGEoZYN5zGkkmGbDmYV9QVNBTJZ+5Cdwx6F6XU7YuOvWfvXtqlRKBNdttp2ST/SztvHuhFHd+3dx0s69pQAFo2GPoHWOZ0beeQlqpllPoXHQS1lrX2YVSAuGjsYWO5eL/1qEV7AiMMSf5T64uGQoXYXexxsWt37Uab1g6H3e6jRzbJ1gHR7a49ZGGKtbK9UFnn7NkXLSzrqlXKdLBuX+6rFi0vri77IFraALlvYvELPjfPLlqXyjToPiBGDXnjnl20L+VQXlNKm3Ac5+yic2k50TTMgkl4c+6xkCXmCwTJIFrHunuphqtY9yQJNLqCuNF58MKmf43dB0BG5r2yrHupbaiSwGP40eVjFUJ6pO+bIC8rKPblzsHQvEfrrMWWPR46+9TcLDRsAFszrQUM1ZlOfPhFOLwkmP8oEgRUEiJdBMzhNa74aTTJbgrbGWxYKO1NeFA8zhCkuCnc4wWIEU6Qkpe28pfzKwpSvhqDnV3JDDbWBACRAJsjJBf8KYVkSnGIyP4Aw5MboBiUwxZlAOGNonoPc1FJv7WUpSuHZmCbkQ4KP4G7APvqTYg/0xjMNAfWckm5FKI3OCvmFT4PYQncZJ9c2AYP3XZH2rTivuao5xKmEc9OZLO9EtT+IIsysNiy0xxIZmnbQPovXJXrsbmZOjc2cgAlGuXyqpNzmwBS55P1oO0cNuVMxmJKCJyhmD3Igx+yWwsRsEDCXPapmKT2GXd5WB2QoGzNADxn/lDCIPk2maXgPZGa3Hrg255zU87qkU8PC5h1eHUkJB3wTuCGkh5d1GHXmLPE9ZtIfOTrkN4edNWL0rE3ifioQpmSqalynUIBqxZC5VQKE6F4lrkte+a2+nep+Xff3RPOFAn6M3hOh3olMC+VBLmlevVnD92RwzLsPoAypiWrPOhd1s9oWjaxHutfNdHc7d2xrLlLCljI6SuUBbDfmHV5jVO3HJU8vnt15n74ZiH5LZNiM0NSKs4EfYh0jq3jkWktDTx8+XM8moYiNf76F+ObxWwL0OzLvycIADw9GiTH3yygcwD6HtmAb3Yw/eWfDBq82Tb4aWUw+vCQrNkkSh1ZJqaPP6h9K0NLVUGmH3r1OsyfpTexZxSi5HNUYKk5tBaQgBZK4rJrFiCv5Jk3Nof2AuIZC79Xf3H65Fm9NGcUPFdLjxEQBKoHZdRZLnfWN0rH4tqk25NK5PZY7KeubKdc0NRlnzRqGIRcZjJ91LCmTnENMVTgFvRKZjbOmBHd1kLCD/uDhLOrpXb2NGYpinJ5UoIBecoOqqMbqG6dbjXrpXqM9NF1NkI/T+gcJiGETUu15pBCxkKrgpXqkxP94dqHzZausaI67VbrAZe4ud/MkS2//aD0FBvAlqyFCmsokcSMy5ElZspcbw2gMtFfSwiXKgjBeOSK/t0BSWxkymJbhi9QuE2RdUUn2AGFjRR7XwYXoCNhuq5bp5ZV/fPn/HUc1C0ZKWSpsVaxnMPN+sirykpEdaWRiB0OPG/fBvNG45inac/pdm36djBUb62u4tzK1rqHVqcTffWO4AkkKatc3ViobPpoiVimdiozexJMfz1sRbH0MlEQOzeS3YVLumm4UazrpnIQ7pMKutURIC1LhH5ZIvRJ7WQHfMU71etaeeALqUEjwZ2Qx6MMUtWYiuM/yUFe0URxitwMZMvfvai/5SCJmk8nqg9ETyJBTYyHcxZOfaafMvnwFGYpH16xWPalXqBQlgNvRMrfv6tfqk1WSExFkpkmsweWe2zKjZVzfw07d+jjUOvh4aH17eFhY212sDKr6S9dhkcuo7qTtdBMvPDkyktXcsXxwmDyRM5QtJRYcj4M3KK14sG9ZFz3Acy66snXKUuq2r7CDIOBd/G4WX+PsxsSSsTYB6WdiWOS6zEVqLUizmrrZqBzNQmcU55L5PJx/tSrHhEpvcaqjqajNzU9JDZyr3Cb53AQ2mJvcwbVhDRfodtpRDC5GNCbf5Lo3tMw1Ul5x3KG8JL7qLpkGJkPz35EEmASMGUDK8B06cCCOFXgRThWt0YbDnzd3EtnIzvr9cd1+XVJvVenu7L6BrbSgNYxPa4fDYrr9mOK1SfMZ4Z0SoZrTGMVVx2aeSagkziWEfFPOElsGyEzZOeVBn0BK3DqPSnOdfJoE9XsRayAn+bJhG6GjusPJcSZJNS0esq+Vc6MallWwRDemepB6oysf3ugK46//XrjSRhiqfy6wch1GwlvcsoQ3Qfu8aAioUyMRqESpz1w6LoVAnVmOJWM76u54sopdIKRcq56N6/YhJzmL0odI/dJkrAbwMlfs3Tz1uPyGRbRK9+0i4y++y6SdgLsRItMJdSIZckBSs7VgF2/X3+YF7kjWeTmqZylo4HOfC5QJ9PiS2oomeZFbHuXMPYPR/IOOMjvbivXr98svCVdYwZIA+OlutX7YDkfBfJ9mazphhF5axUGKnwltxDnzED5oT1EfNmnBIjl2Y9KJwjDd9/ph3uuyyz9jFLA56+nKPJ3rT5zqI2emZYcfRnToCM3kkKSZLCf0qHqh0pfq0NVF+q68Yb/BKsr1Yf2HiKW5fM9Cl76sIoZ16l7cXmLVq2q0jWqhWk6NoUFSeIlJ/baPb521DfiufO6tm6t5u/lyvr58/oGZQoqFW+j1De/Rod7F+XzZcnRyD2O5C7SKbhgh07Ml8V2hX5cr7eHy7LLXp15ytLA28zBDgYHg+HQLrZT2VHlW3o1UGke2z6fZOMfkyDjPVUUWEV0vV5lk0oprx3VcX8tk48OUkxKsK7JypZF71T6O7euG371yt3HpirmH7qva2Eeiimd5/CTYoLid8JGMoaWnVB3oFVAB309Llx/e7RC6Fbh7W6SKtTM3eP5tjApMVnSjoVV3Y0YoqfJ9RQbYfx0hgcComsXsy5PWLcp2/g/Ise6k9cX6hOoy9t4TYQsCvmVUSNcD2XLr9nH+4l9Vm9jVKZDoTER2Z2JSUBZZU6aR2GaPiJCDfGzu1IaNHM3Jz1nbqsG4hz1wIsHTusRNXbyBHy9R5QBPHNu8O/TQwB3u6oxU23yZGu9mtINKCoumI1st3IyB0lBgCzWBjcoWP1uGg2octQtOZv/lBjuzBSqspBlfyVJIDUqJXGX1O6tvm+VXzWKrIJbS7WPlyVr++Qp1E4l+1ylQ5l57N77ivy0vxWmShf55UrI0nTpHPmOjTey6C0bb8Csb7xTxIFqU6C8+9N9Zn3xKMvAErAIOo505e/gdLUtbYOhb8rVn+x1qldnlb6E0u87uw+/Mh4uIqcSklRYiRwdlFx5oymjie6Af20Hpf//TXROYbu/egAdN4nmXG/Gm4ZTXFSrB1Mey4Yjyb0Qu7WMZXXK+sYI6u6W626oE8XoH96emfWAvtlo0vcbtMIX1zHVGW7dU1+pyS87MCHJMknLN2MSfUXziaugtCA/x+S1vbv9qv7u3vTXfp2x/vlH0fg4EwJJeml2Pop3qhTJrz/jYcZMXUSjVqjW0v6ms72N+7T3jin7oHIjqB01WeX9rGozmtVGaz0JRu9l4/ZjKktf+QteFi3W0qOjtCxudl8PiDnOjIWoR1UzmCXeOJiJ9PPnhc661lqpG9+M2b/uQza7+vlS9Vul1W+TLi3sqr4L2EaVPskVv0lN+nTNWgNX3II2REEK94JARDUXflXJpTvOyjwVbNHetlTPWOHSPPml8a3seqjWdSW8XTopsmoTNKz0qCsXEJFLZqy/EK5/HRc2Wt4/w6yHVWull7LjQcgvhpcPPxhH8g/+UIUO88zAwaEnsBCL/l5CTn6Ab6r2p5j0o2u0SzKJ3nsa+a+jtVoXDrfur7D/XhoD/NQ5J9V/wxDjUxPmzHQaRaOS11RGzNzfk5Wb9Vnd+vy5ntENAJGsLyJuk+HMrUpGAkMxgtg3r9xjtfbqtrXYRvbCdUDW+6u8gdLMdt3amkiohF9np/rrqSwnn2dsQtyjK4ksZ/RA6lf5zfhGi5JnGFhrDqke2bIwMXVZQ40cizKl4k9mjpryb2WOmuqv/f8XT9EttQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()